# TurboLLM on Kaggle — raw install (no shortcuts)

This is deliberately **not** the polished [one-click dual-T4 notebook](https://www.kaggle.com/code/sonijisons/turbollm-one-click-dual-t4).
There's no prebuilt CUDA engine dataset and no model pre-attached. It runs exactly the one
command any real user runs — `npx turbollm` — against the **currently published npm package**,
then hands you the URL with nothing else set up. Everything past that point (loading an engine,
getting a model in, Turbo Link) happens in the actual GUI, the same way a brand-new self-hoster
would hit it.

Use this to feel the real first-run friction, not to get a fast working chat.

### Before you press ▸▸ Run All
1. **Settings → Accelerator → GPU T4 × 2** (or × 1 — this notebook doesn't require two)
2. **Settings → Internet → On**

No dataset, no model. Then **Run All** and open the printed URL.

## 1 · Preflight
Soft GPU check (warns, doesn't block — this notebook isn't testing the dual-GPU split) and a
hard internet check, since both the npm install and the tunnel need outbound network. Kaggle's
DNS can be flaky for the first few seconds of a brand-new container, so this retries before
giving up.

In [ ]:
import subprocess, time, urllib.request

smi = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
gpus = [l for l in smi.stdout.strip().splitlines() if l.strip()]
if gpus:
    print("GPU(s):\n" + "\n".join(gpus))
else:
    print("WARNING: no GPU detected — Settings \u2192 Accelerator \u2192 GPU T4 (\u00d71 or \u00d72).\n"
          "Continuing anyway; TurboLLM will still install and serve the GUI, it just won't have a GPU to offer.")

last_err = None
for attempt in range(5):
    try:
        urllib.request.urlopen("https://registry.npmjs.org", timeout=8)
        last_err = None
        break
    except Exception as e:
        last_err = e
        print(f"  internet check attempt {attempt + 1}/5 failed ({e}) \u2014 retrying\u2026")
        time.sleep(5)
if last_err is not None:
    raise SystemExit(f"Internet is OFF \u2014 Settings \u2192 Internet \u2192 On ({last_err})")
print("\nPreflight OK \u2014 internet reaches npm.")

## 2 · Node.js
TurboLLM needs Node ≥ 22 (unflagged `node:sqlite`). Kaggle's base image ships an older one, so
this installs Node 22 from NodeSource only if what's already there is too old.

In [ ]:
import subprocess

def node_major():
    try:
        out = subprocess.run(["node", "-e", "process.stdout.write(String(process.versions.node.split('.').map(Number)[0]))"],
                              capture_output=True, text=True, timeout=10)
        return int(out.stdout.strip() or 0)
    except Exception:
        return 0

if node_major() < 22:
    print("installing Node 22 (NodeSource)\u2026")
    subprocess.run("curl -fsSL https://deb.nodesource.com/setup_22.x | bash -", shell=True, check=True)
    subprocess.run(["apt-get", "install", "-y", "--no-install-recommends", "nodejs"], check=True)
else:
    print("Node already \u2265 22")

subprocess.run(["node", "-v"])

## 3 · Launch — `npx turbollm --tunnel`, nothing else
`--tunnel` is TurboLLM's own built-in way to get a public URL on a rented GPU box (it downloads
`cloudflared` itself, no separate install) — the exact flag documented at
[/docs/cli](https://turbollm.dev/docs/cli). The daemon auto-provisions and prints a one-time
access **Token** the moment the tunnel comes up.

This is the bare npm-published package with no engine and no model attached — the daemon starts
with genuinely nothing configured.

In [ ]:
import os

WORK = os.environ.get("KAGGLE_WORKING", "/kaggle/working")
LOG = f"{WORK}/turbollm.log"
PORT = 6996

# setsid detaches the daemon into its own session so it survives this cell returning \u2014
# real backgrounding lives in the shell, not in `!cmd &`, which Jupyter blocks.
os.system(f'setsid nohup npx --yes turbollm@latest --port {PORT} --tunnel --no-open >"{LOG}" 2>&1 < /dev/null &')
print(f"daemon launching in the background, logging to {LOG}")

In [ ]:
import re, pathlib, time

log = pathlib.Path(LOG)
url = tok = None
for _ in range(90):  # up to 3 min \u2014 first run pulls the npm package + cloudflared
    t = log.read_text(errors="ignore") if log.exists() else ""
    m = re.search(r"Tunnel:\s*(\S+)", t)
    k = re.search(r"Token:\s*(\S+)", t)
    if m: url = m.group(1)
    if k: tok = k.group(1)
    if url and tok:
        break
    time.sleep(2)

print("=" * 60)
print("  OPEN YOUR TURBOLLM GUI")
print("=" * 60)
print("  URL  :", url or "(not found yet \u2014 re-run this cell, or check the log below)")
print("  Token:", tok or "(not found yet \u2014 re-run this cell, or check the log below)")
print("=" * 60)
if not (url and tok):
    print("\nLast 40 lines of", LOG, ":\n")
    print("\n".join((log.read_text(errors="ignore").splitlines() if log.exists() else [])[-40:]))

## Now it's your turn
Open the URL, paste the Token, and go — nothing past this point is scripted. In particular:

- **No engine is set up.** On Linux+NVIDIA the auto-recommendation is Vulkan, and Kaggle's driver
  has no Vulkan ICD, so it'll quietly run on CPU unless you build/attach CUDA yourself
  (Engines → Add engine → Build from source). This is the exact path that used to only offer a
  dead website link for missing build tools — worth noting whether that's still true for you.
- **No model is attached.** Models → Download, or attach a Kaggle dataset via Add Input.
- **Turbo Link**, if you want to try linking this box to another TurboLLM instance, is under
  Settings → Experimental.

Note anything confusing exactly where it happens — that's the point of running it raw.

In [ ]:
# Keep the tunnel alive under a BATCH run (`kaggle kernels push`), which ends the session the
# moment the last cell returns \u2014 an interactive Run All stays up on its own and this loop
# is a no-op there.
import os, re, pathlib, time

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") != "Batch":
    print("Interactive session \u2014 tunnel stays up while this notebook is running.")
else:
    log = pathlib.Path(LOG)
    HOURS = 9
    for minute in range(HOURS * 60):
        if minute % 10 == 0:
            t = log.read_text(errors="ignore") if log.exists() else ""
            m = re.search(r"Tunnel:\s*(\S+)", t)
            k = re.search(r"Token:\s*(\S+)", t)
            print(f"[keep-alive {minute // 60}h{minute % 60:02d}m] "
                  f"URL: {m.group(1) if m else '?'}  Token: {k.group(1) if k else '?'}",
                  flush=True)
        time.sleep(60)